# Yelp Reviews Text Quiz – Practice Notebook

This notebook is a **template** for your quiz.  
It assumes you have a DataFrame of Yelp restaurant reviews in a column called `"review"`.

- Everywhere you see `TODO: ...`, fill in the specific values from the quiz.
- You can also rename variables if the quiz uses different names, but this structure should work.


## 0. Imports and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

# Make plots appear in the notebook (if using Jupyter locally)
%matplotlib inline

# === TODO: Update this path for the quiz CSV file ===
DATA_PATH = "yelp_reviews.csv"  # e.g. something like "quiz_yelp_reviews.csv"

# === TODO: If the text column has a different name, change TEXT_COL ===
TEXT_COL = "review"

df = pd.read_csv(DATA_PATH)

# Quick check
df.head()


## 1. Fix an incorrectly typed location

**Task:** Replace all instances of an incorrectly typed location in the reviews.  
The mistakes come from:
- lack of capitalization  
- a missing dash `"-"`

We will:
1. Define the **incorrect pattern** as a regular expression (`WRONG_LOCATION_PATTERN`).
2. Define the **correct, standardized location string** (`CORRECT_LOCATION`).
3. Use `.str.replace()` with `regex=True` and case-insensitive matching.


In [ ]:
# === TODO: Set these based on the quiz description ===
# Example idea:
#   reviews might contain 'san francisco bay area' instead of 'San-Francisco Bay Area'
WRONG_LOCATION_PATTERN = r"san francisco bay area"  # <-- put the *incorrect* spelling/pattern here
CORRECT_LOCATION = "San-Francisco Bay Area"         # <-- put the *correct* version here

df_q1 = df.copy()

# Replace all case-insensitive matches of WRONG_LOCATION_PATTERN with CORRECT_LOCATION
df_q1[TEXT_COL] = df_q1[TEXT_COL].str.replace(
    WRONG_LOCATION_PATTERN,
    CORRECT_LOCATION,
    case=False,      # ignore capitalization
    regex=True       # treat WRONG_LOCATION_PATTERN as a regex
)

# Quick check of a few rows
df_q1[[TEXT_COL]].head()


## 2. Filter reviews by a string pattern with special regex characters

**Task:** Create a new DataFrame containing **only** the reviews that match a given regex pattern.  
The pattern will involve one or more **special characters** from regular expressions (e.g. `.` `*` `+` `?` `\d` `\w` `[]` etc.).

We will:
1. Set `PATTERN_Q2` to the regex given in the quiz.
2. Use `.str.contains()` with `regex=True` and `na=False` to build a boolean mask.
3. Filter `df` using this mask.


In [ ]:
# === TODO: Set this to the regex pattern from the quiz ===
# Example idea (NOT for the quiz unless it matches!):
# PATTERN_Q2 = r"\d+\s*stars?"  # reviews mentioning "4 stars", "5 star", etc.
PATTERN_Q2 = r"YOUR_REGEX_PATTERN_HERE"

mask_q2 = df[TEXT_COL].str.contains(PATTERN_Q2, regex=True, na=False)
df_q2 = df.loc[mask_q2].copy()

print(f"Number of matching reviews: {mask_q2.sum()}")
df_q2[[TEXT_COL]].head()


## 3. Find two similarly spelled words (singular + plural)

**Task:** In the `review` text, find all occurrences of **two words with similar spellings**, including both plural and singular forms.

Example idea (you will change this for the quiz):
- `color` vs `colour` → include `color`, `colors`, `colour`, `colours`.

We will:
1. Set `WORD1` and `WORD2` to the two base forms (singular) of the words.
2. Build a regex pattern that includes both singular and plural for both words.
3. Use `.str.findall()` to collect all matches for each review.
4. Filter to only the rows that have at least one match.


In [ ]:
# === TODO: Set these based on the quiz ===
# Example idea (NOT necessarily for the quiz):
WORD1 = "color"   # base form of first word (singular)
WORD2 = "colour"  # base form of second word (singular)

# Build a case-insensitive regex that matches singular and plural of both words
# (?i)  -> inline flag for case-insensitive
# \b   -> word boundary
pattern_q3 = rf"(?i)\b({WORD1}s?|{WORD2}s?)\b"

# Find all matches in each review
matches_q3 = df[TEXT_COL].str.findall(pattern_q3)

# Keep only rows where we found at least one match
mask_q3 = matches_q3.str.len() > 0
df_q3 = df.loc[mask_q3, [TEXT_COL]].copy()
df_q3['matches'] = matches_q3[mask_q3]

print(f"Number of reviews with at least one match: {mask_q3.sum()}")
df_q3.head()


## 4. Find words ending with a given letter

**Task:** In the `review` text, find all words whose **last letter** is a specific letter (for example `"y"`).  
Return only the rows where at least one such word is found.

We will:
1. Set `ENDING_LETTER` to the letter given in the quiz.
2. Build a regex pattern that matches any word ending with that letter.
3. Use `.str.findall()` to find all matches in each review.
4. Filter to only rows with at least one match.


In [ ]:
# === TODO: Set this to the target ending letter from the quiz ===
ENDING_LETTER = "y"  # e.g. "y", "e", etc.

# (?i) -> case-insensitive
# \b\w*X\b -> any word of word characters ending with X
pattern_q4 = rf"(?i)\b\w*{ENDING_LETTER}\b"

matches_q4 = df[TEXT_COL].str.findall(pattern_q4)

mask_q4 = matches_q4.str.len() > 0
df_q4 = df.loc[mask_q4, [TEXT_COL]].copy()
df_q4['matches'] = matches_q4[mask_q4]

print(f"Number of reviews with at least one word ending in '{ENDING_LETTER}': {mask_q4.sum()}")
df_q4.head()


## 5. Create a `tokens` column of lower-case word lists

**Task:** Create a new column called `"tokens"` where each review is converted into a list of lower-case words.

We will:
1. Lower-case the text.
2. Use a regex like `\b\w+\b` to extract words (letters, digits, underscore).
3. Store those lists in a new column.


In [ ]:
df_tokens = df.copy()

# Convert to lower-case and split into words using a regex
df_tokens['tokens'] = (
    df_tokens[TEXT_COL]
    .str.lower()
    .str.findall(r"\b\w+\b")
)

df_tokens[['tokens']].head()


## 6. Word probability distribution and bar plot

**Task:**

1. Unravel the `tokens` column into a **single Series** containing all words across all reviews.
2. Compute the **probability distribution** of words:
   - `P(word) = count(word) / total_number_of_words`
3. Create a **bar plot** of the probabilities for a subset of the most common words.

We will:
- Use `.explode()` to go from lists of tokens to a long Series of individual words.
- Use `.value_counts()` to get counts.
- Divide by the total to get probabilities.
- Plot the **top N** words.


In [ ]:
# Make sure df_tokens with 'tokens' column already exists (from Question 5)

# 1. Unravel tokens into a single Series
all_words = df_tokens['tokens'].explode().dropna()

# 2. Compute word counts and probabilities
word_counts = all_words.value_counts()
total_words = word_counts.sum()
word_probs = word_counts / total_words  # This is a Pandas Series: index=word, value=probability

# === TODO: Choose how many of the most common words to plot ===
TOP_N = 20

top_word_probs = word_probs.head(TOP_N)

print("Top word probabilities:")
print(top_word_probs)

# 3. Plot a bar chart of the top word probabilities
plt.figure(figsize=(10, 4))
top_word_probs.plot(kind="bar")
plt.ylabel("Probability")
plt.xlabel("Word")
plt.title(f"Top {TOP_N} Most Probable Words in Yelp Reviews")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
